# MangoLeaf AI: MobileNetV3-Large Transfer Learning Notebook
This notebook contains the complete machine learning pipeline to:
1. Mount Google Drive and load raw leaf & fruit mango dataset folders.
2. Automatically split raw images into 70% Training, 15% Validation, and 15% Testing subsets.
3. Apply 3x pre-augmentation to minority classes (bacterial_black_spot, mango_scab, healthy_fruit) and merge healthy leaf + fruit into 6 target categories.
4. Execute Two-Stage Fine-Tuning on pre-trained MobileNetV3-Large.
5. Export SavedModel and convert to web-optimized TensorFlow.js Graph Model format.

### Step 1: Mount Google Drive
Run this cell to grant Google Colab access to your Drive files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 2: Automated 70% / 15% / 15% Stratified Dataset Splitter
This automatically splits your 7 raw Google Drive folders into dataset/train (70%), dataset/val (15%), and dataset/test (15%).

In [ ]:
import os, shutil
from sklearn.model_selection import train_test_split

raw_dir = '/content/drive/MyDrive/mango_dataset'
output_dir = 'dataset'

folders = ['anthracnose', 'bacterial_black_spot', 'bacterial_canker', 'healthy_fruit', 'healthy_leaf', 'mango_scab', 'sooty_mold']

for folder in folders:
    folder_path = os.path.join(raw_dir, folder)
    if not os.path.exists(folder_path): continue
    images = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
    
    # 70% Train, 30% Temp (Val + Test)
    train_files, temp_files = train_test_split(images, test_size=0.30, random_state=42)
    # Split 30% Temp equally into 15% Val and 15% Test
    val_files, test_files = train_test_split(temp_files, test_size=0.50, random_state=42)
    
    # Copy to target split directories
    for split_name, file_list in [('train', train_files), ('val', val_files), ('test', test_files)]:
        target_class = 'healthy' if 'healthy' in folder else folder
        dst_path = os.path.join(output_dir, split_name, target_class)
        os.makedirs(dst_path, exist_ok=True)
        for f in file_list:
            shutil.copy(os.path.join(folder_path, f), os.path.join(dst_path, f))

print('🎉 70% / 15% / 15% Dataset Split Completed Automatically!')

### Step 3: Apply 3x Pre-Augmentation to Minority Fruit Classes
Applies 3x pre-augmentation to bacterial_black_spot, mango_scab, and healthy_fruit inside the training set.

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

datagen = ImageDataGenerator(rotation_range=30, zoom_range=0.2, horizontal_flip=True, fill_mode='nearest')

fruit_targets = ['bacterial_black_spot', 'mango_scab', 'healthy']
for target in fruit_targets:
    dst = f'dataset/train/{target}'
    if not os.path.exists(dst): continue
    files = [f for f in os.listdir(dst) if not f.startswith('aug3x') and os.path.isfile(os.path.join(dst, f))]
    for filename in files:
        img_path = os.path.join(dst, filename)
        try:
            img = load_img(img_path, target_size=(224, 224))
            x = img_to_array(img)
            x = np.expand_dims(x, axis=0)
            i = 0
            for batch in datagen.flow(x, batch_size=1, save_to_dir=dst, save_prefix='aug3x', save_format='jpg'):
                i += 1
                if i >= 2: break
        except Exception as e:
            continue

print('✅ 3x Pre-Augmentation Complete!')

### Step 4: Data Loaders & Model Construction
Load MobileNetV3-Large base model.

In [ ]:
import tensorflow as tf

train_gen = tf.keras.utils.image_dataset_from_directory('dataset/train', image_size=(224, 224), batch_size=32)
val_gen = tf.keras.utils.image_dataset_from_directory('dataset/val', image_size=(224, 224), batch_size=32)
test_gen = tf.keras.utils.image_dataset_from_directory('dataset/test', image_size=(224, 224), batch_size=32)

base_model = tf.keras.applications.MobileNetV3Large(input_shape=(224, 224, 3), include_top=False, weights='imagenet')

x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(6, activation='softmax')(x)
model = tf.keras.Model(inputs=base_model.input, outputs=outputs)

print('✅ MobileNetV3 Model Ready!')

### Step 5: Two-Stage Fine-Tuning
Stage 1 trains classification head (5 epochs). Stage 2 unfreezes all layers for fine-tuning at learning rate 1e-5 (10 epochs).

In [ ]:
print('🚀 STAGE 1: Training Classification Head (5 Epochs)...')
base_model.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history1 = model.fit(train_gen, validation_data=val_gen, epochs=5)

print('🚀 STAGE 2: Full Fine-Tuning (10 Epochs)...')
base_model.trainable = True
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history2 = model.fit(train_gen, validation_data=val_gen, epochs=10)

### Step 6: Test Set Evaluation
Evaluate final trained model on the clean 15% Test Set.

In [ ]:
test_loss, test_acc = model.evaluate(test_gen)
print(f'\n🏆 FINAL TEST ACCURACY: {test_acc * 100:.2f}%')
print(f'🏆 FINAL TEST LOSS: {test_loss:.4f}')

### Step 7: Export SavedModel & Convert to TensorFlow.js Graph Model

In [ ]:
model.export('mangoleaf_saved_model')

!pip install tensorflowjs
!tensorflowjs_converter --input_format=tf_saved_model mangoleaf_saved_model public/model

print('🎉 Model Conversion Complete! Download files in public/model and paste into off-on-sync/public/model/')